<h1><b><i>SCRIPT PRINCIPAL</b></i></h1>
<h2><b><i>Importação de Bibliotécas</b></i></h2>

In [452]:
import sys
sys.path.append(r'C:\pylibs')

import pandas as pd
import numpy as np
import openpyxl

In [453]:
import pandas as pd
import openpyxl
import numpy as np
import os

In [454]:
# ==============================================================================
# 1. SETUP INICIAL E CARREGAMENTO
#    (Melhor prática: definir constantes e carregar o arquivo de forma segura)
# ==============================================================================

# Definição do caminho do arquivo em uma variável para fácil manutenção
FILE_PATH = 'Planilha de Monitoramento_PPI_2025_31_03_25_PPI_R21.xlsx'
OUTPUT_DIR = 'Dados Gerados'

# Verifica se o arquivo existe antes de tentar carregá-lo
if not os.path.exists(FILE_PATH):
    print(f"ERRO: O arquivo '{FILE_PATH}' não foi encontrado.")
    # Encerra o script ou lança uma exceção se o arquivo principal não existir
    exit()

# Cria a pasta de saída se ela não existir
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Arquivo '{FILE_PATH}' encontrado. Iniciando processamento...")

Arquivo 'Planilha de Monitoramento_PPI_2025_31_03_25_PPI_R21.xlsx' encontrado. Iniciando processamento...


In [455]:
# Listar todas as planilhas disponíveis no arquivo
caminho_arquivo = "Planilha Monitoramento_PPI - Carregamento - BI 2025 - 7.xlsx"
xls = pd.ExcelFile(caminho_arquivo)
sheet_names = xls.sheet_names
sheet_names

['INFORMAÇÕES',
 'PER',
 'PLAN_EXEC (ATÉ 2024)',
 'INEXECUTADO ATÉ 2024 Pendências',
 'A EXECUTAR',
 'META(2025)',
 'META(2026)',
 'META(2027)',
 'Todas Tabelas']

In [456]:
file_path = 'Planilha de Monitoramento_PPI_2025_31_03_25_PPI_R21 - PER.xlsx'

<h1><b><i>PER (1)</b></i></h1>

In [ ]:
# %% =====================================================================
# LEITURA INTELIGENTE DA PLANILHA "PER (1)" – VERSÃO CORRIGIDA
# ========================================================================

import pandas as pd
import numpy as np
from collections import defaultdict

def expandir_mescladas_para_cabecalho(df_cab):
    """Expande horizontalmente (ffill) para simular células mescladas."""
    return df_cab.ffill(axis=1)

def montar_cabecalho_hierarquico(df_cab):
    """
    Recebe um DataFrame com N linhas de cabeçalho (já com ffill)
    e retorna uma lista de nomes de colunas concatenando os níveis.
    """
    lixo = {"#REF!", "TOTAL", "SOMA", 0, "0", None, ""}
    df_cab = df_cab.replace(list(lixo), pd.NA)
    
    colunas = []
    for col in range(df_cab.shape[1]):
        partes = []
        for row in range(df_cab.shape[0]):
            valor = df_cab.iat[row, col]
            if pd.isna(valor):
                continue
            valor = str(valor).strip()
            if not valor or valor.startswith(("=", "#")):
                continue
            if valor.upper() in ("TOTAL", "SOMA"):
                continue
            try:
                float(valor.replace(",", "."))
                continue
            except:
                pass
            partes.append(valor)
        colunas.append(" - ".join(partes) if partes else "")
    
    # Remove duplicatas adicionando sufixo __n
    contador = defaultdict(int)
    novas = []
    for nome in colunas:
        contador[nome] += 1
        if contador[nome] == 1:
            novas.append(nome)
        else:
            novas.append(f"{nome}")
    return novas

# ========================================================================
# 1. DETECTAR A ÚLTIMA COLUNA COM DADOS NO CABEÇALHO
# ========================================================================

file_path = 'Planilha de Monitoramento_PPI_2025_31_03_25_PPI_R21.xlsx'
sheet_name = 'PER (1)'

# Lê apenas as 3 primeiras linhas (cabeçalho) para identificar a última coluna não vazia
cabecalho_raw = pd.read_excel(file_path, sheet_name=sheet_name, header=None, nrows=3)

# Encontra o índice da última coluna que contém algum valor não nulo (em qualquer linha)
ultima_coluna_idx = cabecalho_raw.columns[cabecalho_raw.notna().any(axis=0)].max() + 1  # +1 pois o range é exclusivo
if pd.isna(ultima_coluna_idx):
    ultima_coluna_idx = 0  # fallback

# Número de colunas a serem lidas
num_colunas = int(ultima_coluna_idx)
print(f"🔍 Última coluna com dados: {num_colunas}")

# ========================================================================
# 2. LER CABEÇALHO E DADOS COM O MESMO NÚMERO DE COLUNAS
# ========================================================================

cabecalho_raw = pd.read_excel(
    file_path, sheet_name=sheet_name, header=None, nrows=3, usecols=range(num_colunas)
)

# Expande horizontalmente (simula mesclas)
cabecalho_expandido = expandir_mescladas_para_cabecalho(cabecalho_raw)

# Gera os nomes das colunas a partir das 3 linhas
novos_nomes = montar_cabecalho_hierarquico(cabecalho_expandido)

# Lê os dados pulando as 3 primeiras linhas
df_per = pd.read_excel(
    file_path, sheet_name=sheet_name, header=None, skiprows=3, usecols=range(num_colunas)
)

# Atribui os nomes gerados
df_per.columns = novos_nomes

# Remove linhas totalmente vazias
df_per = df_per.dropna(how='all')

print(f"✅ Planilha PER carregada com {df_per.shape[0]} linhas e {df_per.shape[1]} colunas.")

# ========================================================================
# 3. IDENTIFICAR COLUNAS DE IDENTIFICAÇÃO E PREENCHER PARA BAIXO
# ========================================================================

KEYWORDS_ID = [
    "SETOR", "UF", "ESTADO/LOTE", "BR", "EMPREENDIMENTO", "PROPONENTE",
    "EXECUTOR", "ESTRUTURADOR", "ID-ÚNICO"
]

id_cols = []
for col in df_per.columns:
    col_upper = col.upper()
    if any(kw in col_upper for kw in KEYWORDS_ID):
        id_cols.append(col)

print("Colunas identificadas como ID:")
for c in id_cols:
    print(f"  - {c}")

if id_cols:
    df_per[id_cols] = df_per[id_cols].ffill()
    print("✅ Forward fill aplicado nas colunas de identificação.")

# ========================================================================
# 4. REMOVER LINHAS COM "TOTAL" OU "SOMA" NAS COLUNAS DE MÉTRICA
# ========================================================================

metric_cols = [c for c in df_per.columns if c not in id_cols]

mask_total = df_per[metric_cols].astype(str).apply(
    lambda row: row.str.contains('TOTAL|SOMA', case=False, na=False).any(), axis=1
)
df_per = df_per[~mask_total].reset_index(drop=True)

print(f"✅ Linhas com TOTAL/SOMA removidas. Restam {df_per.shape[0]} linhas.")

# ========================================================================
# 5. MELT (FORMATO LONGO) E SEPARAÇÃO DOS NÍVEIS
# ========================================================================

df_long = df_per.melt(
    id_vars=id_cols,
    value_vars=metric_cols,
    var_name="cabecalho_completo",
    value_name="Valor"
)

df_long = df_long.dropna(subset=["Valor"])
df_long = df_long[df_long["Valor"] != 0]

print(f"✅ Melt aplicado: {df_long.shape[0]} linhas.")

# ========================================================================
# 6. SEPARAR O CABEÇALHO EM NÍVEIS
# ========================================================================

split_headers = df_long["cabecalho_completo"].str.split(" - ", expand=True)
num_niveis = split_headers.shape[1]
split_headers.columns = [f"Nivel_{i+1}" for i in range(num_niveis)]

df_long = pd.concat([df_long, split_headers], axis=1)
df_long.drop(columns=["cabecalho_completo"], inplace=True)

# ========================================================================
# 7. PIVOTAR PARA TER AS MÉTRICAS COMO COLUNAS
# ========================================================================

# Detecta qual nível contém as métricas (Descrição, Ext., Financeiro, (km)%)
niveis_metricas = []
for nivel in [f"Nivel_{i}" for i in range(1, num_niveis+1)]:
    valores = df_long[nivel].dropna().unique()
    if any(k in " ".join(valores.astype(str)) for k in ["Descrição", "Ext", "FINANCEIRO", "(km)%"]):
        niveis_metricas.append(nivel)

nivel_metrica = niveis_metricas[-1] if niveis_metricas else "Nivel_3"
print(f"🔍 Nível identificado como métrica: {nivel_metrica}")

index_cols = id_cols + [c for c in df_long.columns if c.startswith("Nivel_") and c != nivel_metrica]

df_pivot = df_long.pivot_table(
    index=index_cols,
    columns=nivel_metrica,
    values="Valor",
    aggfunc="first"
).reset_index()

# Limpa o nome das colunas (remove o nome do nível, se houver)
df_pivot.columns = [col if col == '' else col for col in df_pivot.columns]

print(f"✅ Pivot concluído: {df_pivot.shape[0]} linhas x {df_pivot.shape[1]} colunas.")

# ========================================================================
# 8. (OPCIONAL) SALVAR
# ========================================================================

df_pivot.to_excel("Dados Gerados/PER_Processada.xlsx", index=False)
print("✅ Arquivo salvo.")

🔍 Última coluna com dados: 229
✅ Planilha PER carregada com 2146 linhas e 229 colunas.
Colunas identificadas como ID:
  - SETOR
  - UF
  - PLANILHA DE MONITORAMENTO - ESTADO/LOTE
  - PLANILHA DE MONITORAMENTO - BR
  - PLANILHA DE MONITORAMENTO - DADOS CONTRATUAIS - EMPREENDIMENTO
  - PLANILHA DE MONITORAMENTO - DADOS CONTRATUAIS - PROPONENTE
  - PLANILHA DE MONITORAMENTO - DADOS CONTRATUAIS - EXECUTOR             (Grupo Controlador)
  - PLANILHA DE MONITORAMENTO - DADOS CONTRATUAIS - ESTRUTURADOR DO PROJETO
✅ Forward fill aplicado nas colunas de identificação.
✅ Linhas com TOTAL/SOMA removidas. Restam 2022 linhas.
✅ Melt aplicado: 24752 linhas.
🔍 Nível identificado como métrica: Nivel_3
✅ Pivot concluído: 2395 linhas x 12 colunas.
✅ Arquivo salvo.
